# ReSnap v5b: contoh rempah dari dataset relz (bawang putih asli + rempah lain sebagai negatif)

**Masalah dari v5a:** di test eksternal, v5a mendeteksi sesuatu di 20.8% foto rempah lain (v4b 9.9%), hampir semuanya
bawang putih dan telur pada tumpukan butiran (lada, kemiri, dll.).

**Satu perubahan dibanding v5a:** data train v5a ditambah foto dari dataset Roboflow `relz-f3q8v/deteksi-rempah-jjceg` (CC BY 4.0):
- bawang putih, bawang merah, bawang bombai, jahe, lengkuas **diberi label** kelas ReSnap;
- rempah lain (lada, cengkeh, kapulaga, pala, adas, jinten, bunga lawang, andaliman, kayu manis, saffron, daun kemangi, daun ketumbar)
  **tidak dilabel**, sehingga menjadi contoh negatif.
Resep training sama persis dengan v5a. Sumber `rival` (test eksternal) tidak disentuh; foto relz yang kembar dengan rival dibuang.

**Validasi:** split valid + test relz (tidak ikut training) untuk (a) deteksi palsu pada foto yang hanya berisi rempah lain dan
(b) AP bawang putih/merah/bombai dalam konteks rempah. Bila validasi negatif kurang dari 30 foto, notebook berhenti dengan pesan jelas.

Syarat v5b diterima (ditetapkan sebelum training, dibanding v5a):
1. Foto validasi negatif yang memunculkan deteksi (conf ≥ 0.5) turun minimal separuh.
2. mAP50-95 val tidak turun lebih dari 0.5 poin.
3. Tidak ada kelas val (≥ 20 objek) yang AP50-nya turun lebih dari 10 poin.
4. AP50 holdout rimpang dan holdout Open Images tidak turun lebih dari 5 poin.
5. Kurva tidak overfit/underfit dan gap train-val tidak lebih dari gap v5a + 0.02.

Kaggle: GPU T4, Internet on, Secret `ROBOFLOW_API_KEY`. Input: output notebook **ReSnap v5a - Data Train** dan dataset **resnap-v4**.
Perkiraan 3 jam. Jalankan dengan Save & Run All (Commit).

In [ ]:
%pip -q install ultralytics==8.3.228 roboflow

In [ ]:
import json, math, random, shutil, subprocess, sys, zipfile
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from PIL import Image, ImageDraw, ImageOps

assert torch.cuda.is_available(), 'GPU tidak aktif. Settings > Accelerator > GPU T4.'
DEVICE = '0'
random.seed(42)
INPUT, OUTPUT, TMP = Path('/kaggle/input'), Path('/kaggle/working'), Path('/tmp/resnap')
OUT = OUTPUT / 'hasil_v5b'
for p in (OUT, TMP):
    p.mkdir(parents=True, exist_ok=True)

REPO = TMP / 'resnap-cv'
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/AffanAbya30/resnap-cv.git', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'scripts'))
import data_v4 as dv
from diagnose_fit import analyze_curves, load_results, plot as plot_curves, verdict
from evaluate_export import evaluate
from ultralytics import YOLO

V4B = REPO / 'models/best.pt'
V5A = next(INPUT.rglob('best_v5a.pt'), None)
assert V5A, 'best_v5a.pt tidak ditemukan: Add Input output notebook "ReSnap v5a - Data Train"'

def first_dir(pred):
    hits = sorted((p for p in INPUT.rglob('*') if p.is_dir() and pred(p)), key=lambda p: len(p.parts))
    return hits[0] if hits else None

D5A = first_dir(lambda p: (p / 'holdout_rimpang/images').is_dir() and (p / 'train/images').is_dir())
if D5A is None:
    z = next(INPUT.rglob('dataset_v5a.zip'), None)
    assert z, 'dataset_v5a.zip tidak ditemukan'
    D5A = TMP / 'd5a'
    if not (D5A / 'train').exists():
        with zipfile.ZipFile(z) as f:
            f.extractall(D5A)
BASE = first_dir(lambda p: (p / 'test/images').is_dir() and (p / 'train/images').is_dir() and not (p / 'holdout_rimpang').exists())
assert BASE, 'Input resnap-v4 belum ditambahkan (hanya untuk cek foto kembar dengan test lama)'
print('dataset v5a:', D5A, '| v3:', BASE)
NAMES = dict(enumerate(dv.CLASS_NAMES)); IDX = {v: k for k, v in NAMES.items()}

def hash_many(paths):
    with ThreadPoolExecutor(8) as ex:
        hs = list(ex.map(lambda p: (dv.dhash(p), dv.dhash(p, mirror=True)), paths))
    return {str(p): h for p, h in zip(paths, hs) if h[0] is not None}

## 1. Unduh relz dan petakan kelas

In [ ]:
from roboflow import Roboflow
import requests
from kaggle_secrets import UserSecretsClient
KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')
SLUG = 'relz-f3q8v/deteksi-rempah-jjceg'
j = requests.get(f'https://api.roboflow.com/{SLUG}', params={'api_key': KEY}, timeout=30).json()
vers = sorted(int(v['id'].split('/')[-1]) for v in j.get('versions', []))
print('versi tersedia:', vers, '| lisensi:', j.get('project', {}).get('license'))
RELZ = TMP / 'relz'
ws, proj = SLUG.split('/')
Roboflow(api_key=KEY).workspace(ws).project(proj).version(vers[-1]).download('yolov8', location=str(RELZ), overwrite=True)
del KEY
shutil.make_archive(str(OUTPUT / 'relz_raw'), 'zip', RELZ)

cfg = yaml.safe_load((RELZ / 'data.yaml').read_text())
names = cfg['names'] if isinstance(cfg['names'], list) else [cfg['names'][k] for k in sorted(cfg['names'])]
MAP = {'bawang putih': 'garlic', 'bawang merah': 'shallot', 'bawang bombai': 'onion', 'bawang bombay': 'onion',
       'jahe': 'ginger', 'lengkuas': 'galangal', 'kunyit': 'turmeric'}
cmap = {i: MAP.get(n.strip().lower().replace('_', ' ').replace('-', ' '), 'negatif') for i, n in enumerate(names)}
print({names[i]: v for i, v in cmap.items()})

def read(lbl):
    out = []
    if lbl.exists():
        for line in lbl.read_text().splitlines():
            v = line.split()
            if len(v) < 5:
                continue
            c, nums = int(v[0]), list(map(float, v[1:]))
            if len(nums) == 4:
                cx, cy, w, h = nums
            else:
                xs, ys = nums[0::2], nums[1::2]
                cx, cy, w, h = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2, max(xs) - min(xs), max(ys) - min(ys)
            out.append((c, cx, cy, w, h))
    return out

items = []
for split in ('train', 'valid', 'test'):
    d = RELZ / split / 'images'
    if not d.is_dir():
        continue
    for p in sorted(d.iterdir()):
        if p.suffix.lower() not in dv.IMG_EXT:
            continue
        raw = read(RELZ / split / 'labels' / f'{p.stem}.txt')
        lines = [f"{IDX[cmap[c]]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}" for c, cx, cy, w, h in raw
                 if cmap.get(c, 'negatif') != 'negatif' and w * h >= 0.0004]
        neg = {names[c] for c, *_ in raw if cmap.get(c, 'negatif') == 'negatif'}
        items.append(dict(split=split, img=p, lines=lines, negatif_saja=not lines and bool(neg), rempah=sorted(neg)))
df = pd.DataFrame(items)
print(df.groupby('split').agg(foto=('img', 'size'), berlabel=('lines', lambda s: sum(bool(x) for x in s)),
                              negatif_saja=('negatif_saja', 'sum')))

## 2. Buang foto kembar (dengan semua data ReSnap, test eksternal rival, dan antar-split relz)

In [ ]:
ref_paths = [p for s in ('train', 'val', 'holdout_oi', 'holdout_rimpang', 'test_eksternal_rimpang')
             for p in (D5A / s / 'images').iterdir() if p.suffix.lower() in dv.IMG_EXT]
ref_paths += [p for p in (BASE / 'test/images').iterdir() if p.suffix.lower() in dv.IMG_EXT]
ref_h = hash_many(ref_paths)
ref = {k: h[0] for k, h in ref_h.items()}
print('hash referensi:', len(ref))
rh = hash_many([it['img'] for it in items])
for it in items:
    it['h'], it['hm'] = rh.get(str(it['img']), (None, None))

log = Counter()
def dup(it, pool, d):
    return bool(pool) and (bool(dv.near_duplicates({'a': it['h']}, pool, d)) or bool(dv.near_duplicates({'a': it['hm']}, pool, d)))

def take(group, name, extra_pool=None):
    kept, own = [], {}
    for it in group:
        if it['h'] is None or (not it['lines'] and not it['negatif_saja']):
            log[f'{name}: kosong/rusak'] += 1; continue
        if dup(it, ref, 6) or (extra_pool and dup(it, extra_pool, 6)):
            log[f'{name}: kembar dengan data lain'] += 1; continue
        if dup(it, own, 4):
            log[f'{name}: salinan augmentasi'] += 1; continue
        own[str(it['img'])] = it['h']; kept.append(it)
    return kept, own

val_relz, vpool = take([it for it in items if it['split'] in ('valid', 'test')], 'val relz')
pool = [it for it in items if it['split'] == 'train']; random.shuffle(pool)
train_pool, _ = take(pool, 'train relz', vpool)
val_neg = [it for it in val_relz if it['negatif_saja']]
print(dict(log))
print('val relz:', len(val_relz), '(negatif saja:', len(val_neg), ') | kandidat train:', len(train_pool))
STOP = None
if len(val_neg) < 30:
    STOP = f'Validasi negatif hanya {len(val_neg)} foto (< 30): tidak cukup untuk menilai v5b. Training dibatalkan.'
    print(STOP)

## 3. Buang foto berisi bahan ReSnap lain yang tak berlabel (dicek v4b, conf 0.7)

In [ ]:
# Bawang putih, telur, dan kelas yang memang dipetakan tidak dihitung: justru itulah kesalahan yang ingin diperbaiki.
IGNORE = {IDX[k] for k in ('garlic', 'egg', 'shallot', 'onion', 'ginger', 'galangal', 'turmeric')}
v4b = YOLO(str(V4B))
if STOP is None:
    for name, group in (('val relz', val_relz), ('train relz', train_pool)):
        keep = []
        for it in group:
            if dv.unlabeled_objects(v4b, str(it['img']), it['lines'], conf=0.7, min_cover=0.5, ignore=IGNORE):
                log[f'{name}: bahan lain tak berlabel'] += 1
            else:
                keep.append(it)
        group[:] = keep
    val_neg = [it for it in val_relz if it['negatif_saja']]
    print(dict(log)); print('val relz:', len(val_relz), '| val negatif:', len(val_neg), '| kandidat train:', len(train_pool))
    if len(val_neg) < 30:
        STOP = f'Validasi negatif tinggal {len(val_neg)} foto (< 30). Training dibatalkan.'; print(STOP)

## 4. Ukur dulu: v4b dan v5a pada validasi negatif

In [ ]:
def fp_rate(weights, its):
    m = YOLO(str(weights)); hit, total, cls = 0, 0, Counter()
    paths = [str(it['img']) for it in its]
    for i in range(0, len(paths), 16):
        for r in m.predict(paths[i:i + 16], conf=0.5, iou=0.7, imgsz=640, device=DEVICE, verbose=False, stream=True):
            n = len(r.boxes); total += n; hit += n > 0
            cls.update(NAMES[int(c)] for c in r.boxes.cls.cpu().numpy())
    del m; torch.cuda.empty_cache()
    return {'foto': len(paths), 'foto_dengan_deteksi': int(hit), 'persen': round(100 * hit / max(1, len(paths)), 1),
            'deteksi_total': int(total), 'kelas': dict(cls.most_common(6))}

FP = {}
if STOP is None:
    FP = {'v4b': fp_rate(V4B, val_neg), 'v5a': fp_rate(V5A, val_neg)}
    print(json.dumps(FP, indent=1))
    if FP['v5a']['persen'] < 5:
        STOP = f"v5a hanya salah di {FP['v5a']['persen']}% foto validasi negatif: masalahnya tidak muncul di relz. Training dibatalkan."
        print(STOP)

## 5. Susun dataset v5b dan training (resep sama persis dengan v5a)

In [ ]:
RUN_PROJECT, RUN_NAME = OUTPUT / 'training', 'resnap_yolov8s_v5b_rempah'
RUN_DIR = RUN_PROJECT / RUN_NAME
D5B = TMP / 'data_v5b'
if STOP is None:
    CAP_BOX, CAP_NEG = 400, 500
    added, boxes, neg = [], Counter(), 0
    for it in train_pool:
        cls = Counter(int(l.split()[0]) for l in it['lines'])
        if not cls:
            if neg < CAP_NEG:
                added.append(it); neg += 1
        elif any(boxes[k] < CAP_BOX for k in cls):
            added.append(it); boxes.update(cls)
    print('tambahan train:', len(added), '| kotak:', {NAMES[k]: v for k, v in boxes.items()}, '| negatif:', neg)
    if D5B.exists():
        shutil.rmtree(D5B)
    for s in ('train', 'val', 'holdout_oi', 'holdout_rimpang'):
        shutil.copytree(D5A / s, D5B / s)
    for i, it in enumerate(added):
        dv.write_pair(it['img'], it['lines'], D5B / 'train/images', D5B / 'train/labels', f'rz_{i:05d}')
    for i, it in enumerate(val_relz):
        dv.write_pair(it['img'], it['lines'], D5B / 'val_relz/images', D5B / 'val_relz/labels', f'vr_{i:05d}')
    Y_TRAIN = dv.write_yaml(D5B / 'data.yaml', D5B, 'train/images', 'val/images')
    Y_HOI = dv.write_yaml(D5B / 'hoi.yaml', D5B, 'holdout_oi/images', 'holdout_oi/images')
    Y_HR = dv.write_yaml(D5B / 'hr.yaml', D5B, 'holdout_rimpang/images', 'holdout_rimpang/images')
    Y_VR = dv.write_yaml(D5B / 'vr.yaml', D5B, 'val_relz/images', 'val_relz/images')
    print('train v5b:', dv.count_split(D5B / 'train')['images'], 'foto')

    MAX_EPOCHS = 70
    RECIPE = dict(
        epochs=MAX_EPOCHS, patience=15, batch=16, imgsz=640, save=True, save_period=-1, cache=False,
        workers=2, pretrained=True, optimizer='AdamW', lr0=0.0003, lrf=0.01, momentum=0.937,
        weight_decay=0.001, warmup_epochs=3.0, seed=42, deterministic=True, cos_lr=False,
        close_mosaic=10, amp=True, plots=True, val=True,
        hsv_h=0.015, hsv_s=0.5, hsv_v=0.35, degrees=10.0, translate=0.1, scale=0.5, shear=0.0,
        perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=1.0, mixup=0.1, copy_paste=0.0,
    )
    YOLO('yolov8s.pt').train(data=str(Y_TRAIN), project=str(RUN_PROJECT), name=RUN_NAME, exist_ok=True, device=DEVICE, **RECIPE)
    V5B = RUN_DIR / 'weights/best.pt'
    print('Selesai:', V5B)

## 6. Bandingkan v4b, v5a, v5b

In [ ]:
if STOP is None:
    rng = random.Random(42)
    sub = TMP / 'train_subset'
    if sub.exists():
        shutil.rmtree(sub)
    (sub / 'images').mkdir(parents=True); (sub / 'labels').mkdir()
    old = sorted(p for p in (D5A / 'train/images').iterdir() if not p.stem.startswith('rp_'))
    for p in rng.sample(old, 600):
        shutil.copy2(p, sub / 'images' / p.name)
        shutil.copy2(D5A / 'train/labels' / f'{p.stem}.txt', sub / 'labels' / f'{p.stem}.txt')
    Y_SUB = dv.write_yaml(sub / 'data.yaml', sub, 'images', 'images')

    MODELS = {'v4b': V4B, 'v5a': V5A, 'v5b': V5B}
    E = {}
    for n, w in MODELS.items():
        for tag, y in (('val', Y_TRAIN), ('train_subset', Y_SUB), ('holdout_oi', Y_HOI), ('holdout_rimpang', Y_HR), ('val_relz', Y_VR)):
            E[f'{n}_{tag}'] = evaluate(str(w), y, 'val', OUT, f'{n}_{tag}', DEVICE)
    FP['v5b'] = fp_rate(V5B, val_neg)
    gap = {n: E[f'{n}_train_subset']['mAP50_95'] - E[f'{n}_val']['mAP50_95'] for n in MODELS}
    pc = lambda name: pd.read_csv(OUT / name / 'per_class.csv').set_index('class_name')

    overall = pd.DataFrame({n: E[f'{n}_val'] for n in MODELS}).loc[['precision', 'recall', 'f1', 'mAP50', 'mAP50_95']]
    overall.loc['gap_train_val'] = [gap[n] for n in MODELS]
    overall.loc['fp_val_negatif_persen'] = [FP[n]['persen'] for n in MODELS]
    overall['v5b_vs_v5a_poin'] = 100 * (overall.v5b - overall.v5a)
    display(overall.style.format(precision=3).set_caption('Val 1.069 foto + deteksi palsu validasi negatif'))
    pv = {n: pc(f'{n}_val') for n in MODELS}
    cmp = pd.DataFrame({'objek': pv['v5b']['instances'], **{f'AP50_{n}': pv[n]['AP50'] for n in MODELS}})
    cmp['v5b_vs_v5a'] = 100 * (cmp.AP50_v5b - cmp.AP50_v5a)
    display(cmp.sort_values('v5b_vs_v5a').style.format(precision=3).set_caption('AP50 per kelas, val'))
    hold = lambda tag, cl: pd.DataFrame({n: pc(f'{n}_{tag}').reindex(cl)['AP50'].fillna(0) for n in MODELS}).assign(
        v5b_vs_v5a=lambda t: 100 * (t.v5b - t.v5a))
    hr = hold('holdout_rimpang', ['ginger', 'turmeric', 'galangal'])
    hoi = hold('holdout_oi', ['potato', 'shrimp', 'mushroom'])
    vr = hold('val_relz', ['garlic', 'shallot', 'onion', 'ginger', 'galangal'])
    for t, cap in ((hr, 'Holdout rimpang'), (hoi, 'Holdout Open Images'), (vr, 'Validasi relz (konteks rempah)')):
        display(t.style.format(precision=3).set_caption(cap))
    rows_ = load_results(RUN_DIR / 'results.csv')
    curves = analyze_curves(rows_, MAX_EPOCHS)
    fit = verdict(curves, E['v5b_train_subset']['mAP50_95'], E['v5b_val']['mAP50_95'], ref_gap=gap['v5a'] - 0.03)
    plot_curves(rows_, OUT / 'kurva_v5b.png', curves['best_epoch'], 'Kurva training v5b (rempah relz)')
    for name, d in (('banding_val', overall), ('banding_per_kelas_val', cmp), ('banding_holdout_rimpang', hr),
                    ('banding_holdout_oi', hoi), ('banding_val_relz', vr)):
        d.to_csv(OUT / f'{name}.csv')

## 7. Keputusan dan simpan hasil

In [ ]:
if STOP is None:
    checks = {
        '1_fp_negatif_turun_separuh': FP['v5b']['foto_dengan_deteksi'] <= 0.5 * FP['v5a']['foto_dengan_deteksi'],
        '2_val_map50_95_tidak_turun': overall.loc['mAP50_95', 'v5b_vs_v5a_poin'] >= -0.5,
        '3_tidak_ada_kelas_val_anjlok': not ((cmp.objek >= 20) & (cmp.v5b_vs_v5a < -10)).any(),
        '4_holdout_tidak_turun': bool((hr.v5b_vs_v5a >= -5).all() and (hoi.v5b_vs_v5a >= -5).all()),
        '5_fit_seimbang_gap_terjaga': fit['vonis'] == 'seimbang',
    }
    for k, v in checks.items():
        print(('LULUS ' if v else 'GAGAL ') + k)
    ACCEPT = all(checks.values())
    print('v5b', 'DITERIMA' if ACCEPT else 'BELUM DITERIMA')
    dv.save_json(OUT / 'keputusan_v5b.json', {'diterima': ACCEPT, 'syarat': {k: bool(v) for k, v in checks.items()},
                                             'fp_val_negatif': FP, 'gap': {k: round(v, 4) for k, v in gap.items()},
                                             'diagnosa': fit, 'evaluasi': E, 'log_data': dict(log),
                                             'jumlah': {'tambahan_train': len(added), 'val_relz': len(val_relz), 'val_negatif': len(val_neg)}})
    shutil.copy2(V5B, OUTPUT / 'best_v5b.pt')
    for f in ('results.csv', 'results.png'):
        if (RUN_DIR / f).exists():
            shutil.copy2(RUN_DIR / f, OUT / f'train_{f}')
    shutil.rmtree(RUN_DIR / 'weights', ignore_errors=True)
    shutil.make_archive(str(OUTPUT / 'dataset_v5b'), 'zip', D5B)
else:
    dv.save_json(OUT / 'keputusan_v5b.json', {'dibatalkan': True, 'alasan': STOP, 'fp_val_negatif': FP, 'log_data': dict(log)})
pack = OUTPUT / 'hasil_v5b.zip'
with zipfile.ZipFile(pack, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in OUT.rglob('*'):
        if p.is_file() and p.suffix in ('.json', '.csv', '.png') and 'batch' not in p.name:
            z.write(p, p.relative_to(OUT))
print(pack, round(pack.stat().st_size / 1e6, 2), 'MB')